<a href="https://colab.research.google.com/github/jasonmcgovern/gallery/blob/main/semRushCompetitorComparison_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Import Libraries

import pandas as pd
import numpy as np
import glob
import os
import re
from google.colab import files

In [ ]:
# @title Initialize Script with Configuration

# CONFIGURATION: Set minimum brand count threshold (keywords below this will be removed)
MIN_BRAND_COUNT = 2  # Change this value as needed

def upload_csv_files():
    """
    Creates a file upload widget for CSV files.
    Returns a dictionary of uploaded files.
    """
    print("Please upload your CSV files...")
    uploaded = files.upload()

    if not uploaded:
        print("No files were uploaded.")
        return None

    print(f"Uploaded {len(uploaded)} files.")
    return uploaded

def extract_brand_name(filename):
    """
    Extracts a clean brand name from a filename.
    """
    # Remove file extension and common suffixes like _top20nb
    brand_name = re.sub(r'[_-]top\d+n?b?\.csv$', '', filename, flags=re.IGNORECASE)
    if brand_name.endswith('.csv'):
        brand_name = brand_name[:-4]

    # Capitalize first letter of each word
    brand_name = ''.join(word.capitalize() for word in re.split(r'[_-]', brand_name))

    return brand_name

def process_ranking_data(brand_mapping=None, output_filename='combined_rankings.csv', min_brand_count=MIN_BRAND_COUNT):
    """
    Combines CSV files from different brands, removes duplicate keywords,
    and shows how each brand ranks for every keyword.

    Parameters:
    - brand_mapping: Dictionary mapping brand names to filenames.
                    If None, will use all uploaded CSV files.
    - output_filename: Name of the output CSV file.
    - min_brand_count: Minimum number of brands a keyword must appear in to be included.

    Returns:
    - A pandas DataFrame with combined data
    - Saves the result to the specified output filename
    """
    # If no brand mapping is provided, use all uploaded CSV files
    if brand_mapping is None:
        # Get all CSV files in the current directory
        csv_files = glob.glob('*.csv')

        if not csv_files:
            print("No CSV files found. Please upload some files first.")
            return None

        # Create a brand mapping from filenames
        brand_mapping = {}
        for filename in csv_files:
            brand_name = extract_brand_name(filename)
            brand_mapping[brand_name] = filename

    print(f"Processing data for the following brands: {', '.join(brand_mapping.keys())}")

    # Create a dictionary to store dataframes
    brand_dfs = {}

    # Read each CSV file and add a brand column
    for brand, filename in brand_mapping.items():
        if os.path.exists(filename):
            try:
                df = pd.read_csv(filename)

                # Check if the file has the expected structure
                if 'Keyword' not in df.columns or 'Position' not in df.columns:
                    print(f"Warning: {filename} doesn't have the expected columns (Keyword, Position). Skipping.")
                    continue

                # Filter out positions greater than 30
                original_count = len(df)
                df = df[df['Position'] <= 30]
                filtered_count = original_count - len(df)

                # Remove duplicate keywords, keeping the one with the lowest position
                df = df.sort_values('Position').drop_duplicates(subset='Keyword', keep='first')
                duplicate_count = len(df[df['Position'] <= 30]) - len(df) if filtered_count == 0 else original_count - filtered_count - len(df)

                if filtered_count > 0 or duplicate_count > 0:
                    print(f"{brand}: Removed {filtered_count} rows with position >30 and {duplicate_count} duplicate keywords")

                # Add the brand name
                df['Brand'] = brand

                # Store in dictionary
                brand_dfs[brand] = df

                print(f"Successfully loaded {brand} data: {len(df)} rows")
            except Exception as e:
                print(f"Error reading {filename}: {e}")
        else:
            print(f"File not found: {filename}")

    # Combine all dataframes
    if not brand_dfs:
        print("No data files were successfully loaded. Please check file formats.")
        return None

    # Get all unique keywords across all brands
    all_keywords = set()
    for df in brand_dfs.values():
        all_keywords.update(df['Keyword'].unique())

    print(f"Total unique keywords across all brands: {len(all_keywords)}")

    # Create a new dataframe with unique keywords
    result_df = pd.DataFrame({'Keyword': list(all_keywords)})

    # Add columns for each brand's position AND URL
    for brand, df in brand_dfs.items():
        # Create a mapping from keyword to position
        keyword_to_position = dict(zip(df['Keyword'], df['Position']))

        # Add a column for this brand's position
        result_df[f'{brand}_Position'] = result_df['Keyword'].map(keyword_to_position)

        # NEW: Add URL column if it exists in the source data
        if 'URL' in df.columns:
            keyword_to_url = dict(zip(df['Keyword'], df['URL']))
            result_df[f'{brand}_URL'] = result_df['Keyword'].map(keyword_to_url)

    # Add columns for additional metrics that might be useful (if they exist in the input data)
    common_metrics = [
        'Search Volume', 'Keyword Difficulty', 'CPC',
        'Traffic', 'Traffic (%)', 'Traffic Cost', 'Competition', 'Keyword Intents'
    ]

    # Create standardized column names
    metric_mapping = {
        'Search Volume': 'Search_Volume',
        'Keyword Difficulty': 'Keyword_Difficulty',
        'CPC': 'CPC',
        'Traffic': 'Traffic',
        'Traffic (%)': 'Traffic_Percent',
        'Traffic Cost': 'Traffic_Cost',
        'Competition': 'Competition',
        'Keyword Intents': 'Keyword_Intents'
    }

    # Add metrics data
    for metric, result_col in metric_mapping.items():
        for brand, df in brand_dfs.items():
            if metric in df.columns:
                keyword_to_metric = dict(zip(df['Keyword'], df[metric]))

                if result_col not in result_df.columns:
                    # Initialize the column
                    result_df[result_col] = result_df['Keyword'].map(keyword_to_metric)
                else:
                    # Fill in missing values from this brand
                    metric_map = result_df['Keyword'].map(keyword_to_metric)
                    result_df[result_col] = result_df[result_col].fillna(metric_map)

    # Calculate brand count for each keyword (how many brands rank for this keyword)
    position_columns = [col for col in result_df.columns if col.endswith('_Position')]
    result_df['Brand_Count'] = result_df[position_columns].notna().sum(axis=1)

    # NEW: Filter out keywords below the minimum brand count threshold
    original_count = len(result_df)
    result_df = result_df[result_df['Brand_Count'] >= min_brand_count]
    filtered_count = original_count - len(result_df)
    print(f"\nFiltered out {filtered_count} keywords with brand count below {min_brand_count}")

    # NEW: Sort by Brand_Count in descending order first, then by Search_Volume if available
    if 'Search_Volume' in result_df.columns:
        result_df = result_df.sort_values(by=['Brand_Count', 'Search_Volume'], ascending=[False, False])
    else:
        result_df = result_df.sort_values(by='Brand_Count', ascending=False)

    # Add estimated traffic based on CTR for each position
    # CTR estimates based on typical click-through rates by SERP position
    ctr_by_position = {
        1: 0.316,  # 31.6% CTR for position 1
        2: 0.148,  # 14.8% CTR for position 2
        3: 0.090,  # 9.0% CTR for position 3
        4: 0.064,  # 6.4% CTR for position 4
        5: 0.041,  # 4.1% CTR for position 5
        6: 0.036,  # 3.6% CTR for position 6
        7: 0.029,  # 2.9% CTR for position 7
        8: 0.022,  # 2.2% CTR for position 8
        9: 0.018,  # 1.8% CTR for position 9
        10: 0.016, # 1.6% CTR for position 10
        11: 0.004, # 0.4% CTR for positions 11-20
        12: 0.004,
        13: 0.003,
        14: 0.003,
        15: 0.003,
        16: 0.002,
        17: 0.002,
        18: 0.002,
        19: 0.002,
        20: 0.002
    }

    # Default CTR for positions beyond 20
    default_ctr = 0.001  # 0.1% for positions beyond 20

    # Add estimated traffic columns for each brand
    if 'Search_Volume' in result_df.columns:
        for brand_col in position_columns:
            brand_name = brand_col.replace('_Position', '')
            traffic_col = f'{brand_name}_Est_Traffic'

            # Function to calculate estimated traffic
            def calc_traffic(row):
                position = row[brand_col]
                search_vol = row['Search_Volume']

                if pd.isna(position) or pd.isna(search_vol):
                    return 0

                position = int(position)
                # Get CTR based on position, use default for positions beyond 20
                ctr = ctr_by_position.get(position, default_ctr)

                # Calculate estimated traffic
                return round(search_vol * ctr)

            # Apply calculation to each row
            result_df[traffic_col] = result_df.apply(calc_traffic, axis=1)

        # Calculate total traffic values (just once, not as repeated columns)
        brand_totals = {}
        for brand_col in position_columns:
            brand_name = brand_col.replace('_Position', '')
            traffic_col = f'{brand_name}_Est_Traffic'
            brand_totals[brand_name] = result_df[traffic_col].sum()

        # Store the totals in a dictionary that can be accessed later
        # This avoids adding redundant columns to the dataframe
        result_df.attrs['brand_traffic_totals'] = brand_totals

    # Save to CSV
    result_df.to_csv(output_filename, index=False)
    print(f"Combined data saved to {output_filename}")

    return result_df

def display_statistics(result_df):
    """
    Displays statistics about the combined ranking data.
    """
    if result_df is None or result_df.empty:
        print("No data to analyze.")
        return

    print("\nStatistics:")
    position_columns = [col for col in result_df.columns if col.endswith('_Position')]
    traffic_columns = [col for col in result_df.columns if col.endswith('_Est_Traffic')]

    # Keywords ranked by all brands
    all_brands_count = (result_df[position_columns].notna().sum(axis=1) == len(position_columns)).sum()
    print(f"Keywords ranked by all brands: {all_brands_count}")

    # Keywords ranked by only one brand
    one_brand_count = (result_df[position_columns].notna().sum(axis=1) == 1).sum()
    print(f"Keywords ranked by only one brand: {one_brand_count}")

    # NEW: Show brand count distribution
    print("\nBrand Count Distribution:")
    print(result_df['Brand_Count'].value_counts().sort_index(ascending=False))

    # Show estimated traffic by brand
    if traffic_columns:
        print("\nEstimated Traffic by Brand:")

        # Use the pre-calculated totals if available
        if hasattr(result_df, 'attrs') and 'brand_traffic_totals' in result_df.attrs:
            brand_traffic = result_df.attrs['brand_traffic_totals']
        else:
            # Calculate totals if not already available
            brand_traffic = {}
            for col in traffic_columns:
                brand = col.replace('_Est_Traffic', '')
                total_traffic = result_df[col].sum()
                brand_traffic[brand] = total_traffic

        # Sort brands by traffic and display
        for brand, traffic in sorted(brand_traffic.items(), key=lambda x: x[1], reverse=True):
            print(f"{brand}: {int(traffic):,} estimated monthly visits")

    # Top 10 keywords by search volume that have all brands ranking
    if 'Search_Volume' in result_df.columns and all_brands_count > 0:
        all_brands_df = result_df[result_df[position_columns].notna().all(axis=1)]
        if not all_brands_df.empty:
            top_keywords = all_brands_df.nlargest(10, 'Search_Volume')

            print("\nTop 10 keywords by search volume that all brands rank for:")
            for i, row in top_keywords.iterrows():
                positions = []
                for brand_col in position_columns:
                    brand = brand_col.split('_')[0]
                    position = int(row[brand_col])

                    # Add estimated traffic if available
                    traffic_col = f"{brand}_Est_Traffic"
                    if traffic_col in row:
                        traffic = int(row[traffic_col])
                        positions.append(f"{brand}: #{position} ({traffic:,} visits)")
                    else:
                        positions.append(f"{brand}: #{position}")

                positions_str = ", ".join(positions)
                print(f"{row['Keyword']} (SV: {int(row['Search_Volume']):,}): {positions_str}")

In [ ]:
# @title Upload Files and Process Data

# Upload files
uploaded = upload_csv_files()

# Process data with the configured minimum brand count
result_df = process_ranking_data(min_brand_count=MIN_BRAND_COUNT)

# Display statistics
if result_df is not None:
    display_statistics(result_df)

# Download the output file
if result_df is not None:
    files.download('combined_rankings.csv')